# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

In [6]:
citations_header = rddCitations.first()
citations_data = rddCitations.filter(lambda l: l != citations_header) \
    .map(lambda l: l.split(",")) \
    .map(lambda fields: [f.strip('"') for f in fields])

citations_data.take(5)

[['3858241', '956203'],
 ['3858241', '1324234'],
 ['3858241', '3398406'],
 ['3858241', '3557384'],
 ['3858241', '3634889']]

In [7]:
patents_header = rddPatents.first()
patents_data = rddPatents.filter(lambda l: l != patents_header) \
    .map(lambda l: l.split(",")) \
    .map(lambda fields: [f.strip('"') for f in fields])

patents_data.take(5)

[['3070801',
  '1963',
  '1096',
  '',
  'BE',
  '',
  '',
  '1',
  '',
  '269',
  '6',
  '69',
  '',
  '1',
  '',
  '0',
  '',
  '',
  '',
  '',
  '',
  '',
  ''],
 ['3070802',
  '1963',
  '1096',
  '',
  'US',
  'TX',
  '',
  '1',
  '',
  '2',
  '6',
  '63',
  '',
  '0',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  '',
  ''],
 ['3070803',
  '1963',
  '1096',
  '',
  'US',
  'IL',
  '',
  '1',
  '',
  '2',
  '6',
  '63',
  '',
  '9',
  '',
  '0.3704',
  '',
  '',
  '',
  '',
  '',
  '',
  ''],
 ['3070804',
  '1963',
  '1096',
  '',
  'US',
  'OH',
  '',
  '1',
  '',
  '2',
  '6',
  '63',
  '',
  '3',
  '',
  '0.6667',
  '',
  '',
  '',
  '',
  '',
  '',
  ''],
 ['3070805',
  '1963',
  '1096',
  '',
  'US',
  'CA',
  '',
  '1',
  '',
  '2',
  '6',
  '63',
  '',
  '1',
  '',
  '0',
  '',
  '',
  '',
  '',
  '',
  '',
  '']]

Join citations to patents on CITING patents, to get state of CITING Patent

In [8]:
citing_with_state = citations_data.map(lambda f: (f[0], f[1])) \
    .join(patents_data.map(lambda f: (f[0], f[5])))

print(citing_with_state.take(10))

[('3858416', ('3324685', 'NC')), ('3858416', ('3327499', 'NC')), ('3858416', ('3680753', 'NC')), ('3858416', ('3781532', 'NC')), ('3858683', ('3006434', 'PA')), ('3858683', ('3078955', 'PA')), ('3858729', ('1506811', 'IN')), ('3858729', ('3568857', 'IN')), ('3858729', ('3612296', 'IN')), ('3858826', ('777140', 'MI'))]


Join citations to patents on CITED, to get state of CITED Patent

In [9]:
cited_with_state = citations_data.map(lambda f: (f[1], f[0])) \
    .join(patents_data.map(lambda f: (f[0], f[5])))

print(cited_with_state.take(10))

[('3685428', ('3858491', 'IA')), ('3685428', ('3905284', 'IA')), ('3465217', ('3858509', 'IA')), ('3465217', ('3904946', 'IA')), ('3465217', ('3921305', 'IA')), ('3465217', ('3924721', 'IA')), ('3465217', ('3957151', 'IA')), ('3465217', ('3962620', 'IA')), ('3465217', ('4028604', 'IA')), ('3465217', ('4039067', 'IA'))]


Creating a Key-Value tables to help with the join
- Citing Table's schema: ((CITING, CITED), CITING_STATE)
- Cited Table's schema: ((CITING, CITED), CITED_STATE)

In [10]:
citing_keyed = citing_with_state.map(lambda kv: ((kv[0], kv[1][0]), kv[1][1]))

In [11]:
cited_keyed = cited_with_state.map(lambda kv: ((kv[1][0], kv[0]), kv[1][1]))

Recombine above 2 tables, so each row has CITING & CITED states side by side
- Citing & Cited Table's schema: ((CITING, CITED), (CITING_STATE, CITED_STATE))

In [12]:
citing_and_cited = citing_keyed.join(cited_keyed)

print(citing_and_cited.take(10))

[(('4146877', '3877007'), ('MI', 'MA')), (('4766433', '3683239'), ('CA', 'CA')), (('5858096', '4503804'), ('', 'WI')), (('5174616', '4379575'), ('', 'CA')), (('5412812', '4541125'), ('MO', 'MI')), (('5645528', '4224929'), ('MN', '')), (('5030894', '4645979'), ('', '')), (('5802924', '3446391'), ('IN', 'CA')), (('5009883', '4296096'), ('CA', 'NJ')), (('5909253', '5646698'), ('NJ', 'PA'))]


Keep rows where the CITING & CITED rows match, drop blank rows

In [13]:
same_state = citing_and_cited.filter(
    lambda kv: kv[1][0] == kv[1][1] and kv[1][0]
)

print(same_state.take(10))

[(('4744121', '4110860'), ('WI', 'WI')), (('5038663', '3133725'), ('CA', 'CA')), (('4155443', '4026409'), ('MI', 'MI')), (('5204399', '4652598'), ('NY', 'NY')), (('4575684', '3982195'), ('IL', 'IL')), (('4662547', '4176770'), ('CA', 'CA')), (('4902539', '3150828'), ('IN', 'IN')), (('5682106', '4881114'), ('CA', 'CA')), (('4889078', '4640229'), ('IN', 'IN')), (('4664126', '4140110'), ('CA', 'CA'))]


Count same state citations per citing patent

In [14]:
same_state_counts = same_state.map(lambda kv: (kv[0][0], 1)) \
    .reduceByKey(lambda a, b: a + b)

print(same_state_counts.take(10))

[('4833029', 6), ('5422268', 3), ('5263163', 6), ('5023221', 8), ('5288709', 1), ('4201558', 3), ('5165491', 11), ('5596763', 13), ('5515221', 2), ('4760974', 2)]


Join the counts back into the full patents dataset

In [15]:
augmented = patents_data.map(lambda f: (f[0], f)) \
    .leftOuterJoin(same_state_counts) \
    .map(lambda kv: (kv[0], kv[1][0], kv[1][1] if kv[1][1] is not None else 0))

Top 10 counts sorted in descending order, by count

In [16]:
header_fields = patents_header.split(",")
print(*header_fields, "SAME_STATE")

top10 = augmented.sortBy(lambda x: x[2], ascending=False).take(10)

for patent, record, count in top10:
    print(*record, count)

"PATENT" "GYEAR" "GDATE" "APPYEAR" "COUNTRY" "POSTATE" "ASSIGNEE" "ASSCODE" "CLAIMS" "NCLASS" "CAT" "SUBCAT" "CMADE" "CRECEIVE" "RATIOCIT" "GENERAL" "ORIGINAL" "FWDAPLAG" "BCKGTLAG" "SELFCTUB" "SELFCTLB" "SECDUPBD" "SECDLWBD" SAME_STATE
5959466 1999 14515 1997 US CA 5310 2  326 4 46 159 0 1  0.6186  4.8868 0.0455 0.044   125
5983822 1999 14564 1998 US TX 569900 2  114 5 55 200 0 0.995  0.7201  12.45 0 0   103
6008204 1999 14606 1998 US CA 749584 2  514 3 31 121 0 1  0.7415  5 0.0085 0.0083   100
5952345 1999 14501 1997 US CA 749584 2  514 3 31 118 0 1  0.7442  5.1102 0 0   98
5998655 1999 14585 1998 US CA  1  560 1 14 114 0 1  0.7387  5.1667     96
5958954 1999 14515 1997 US CA 749584 2  514 3 31 116 0 1  0.7397  5.181 0 0   96
5936426 1999 14466 1997 US CA 5310 2  326 4 46 178 0 1  0.58  11.2303 0.0765 0.073   94
5951547 1999 14501 1997 US CA 733846 2  606 3 32 242 0 1  0.7382  8.3471 0 0   90
5739256 1998 13983 1995 US CA 70060 2 15 528 1 15 453 0 1  0.8232  15.1104 0.1124 0.1082   9